# Notebook 02: Data Cleaning & Feature Engineering

## Objective

The objective of this notebook is to prepare the dataset for machine learning by:

- Cleaning unnecessary information
- Encoding categorical variables
- Creating meaningful engineered features
- Verifying the processed dataset

The transformed dataset will be used in the next notebook for model training and evaluation.

In [2]:
# ============================================================
# Smart Manufacturing Data Analytics and Predictive Maintenance
# Notebook 02: Data Cleaning & Feature Engineering
# Author: Opurva Saini
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

In [3]:
df = pd.read_csv("../dataset/ai4i2020.csv")

print("Dataset Loaded Successfully!")

Dataset Loaded Successfully!


In [4]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [5]:
df.shape

(10000, 14)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

### Observation

The dataset has been loaded successfully.

Before building a machine learning model, unnecessary attributes and categorical variables must be processed appropriately.

In [7]:
df.drop(
    columns=["UDI", "Product ID"],
    inplace=True
)

df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


### Why remove these columns?

- **UDI** is simply a serial number.
- **Product ID** uniquely identifies each machine.

These columns contain no predictive information and may introduce unnecessary noise into the machine learning model.

In [8]:
df["Type"].value_counts()

Type
L    6000
M    2997
H    1003
Name: count, dtype: int64

In [9]:
encoder = LabelEncoder()

df["Type"] = encoder.fit_transform(df["Type"])

df["Type"].head()

0    2
1    1
2    1
3    1
4    1
Name: Type, dtype: int64

### Why Encoding?

Machine learning algorithms work with numerical values.

The categorical machine types are converted as:

- L → 0
- M → 1
- H → 2

In [10]:
#Feature Engineering

df["Temperature Difference"] = (
    df["Process temperature [K]"]
    - df["Air temperature [K]"]
)

#Why? Instead of absolute temperatures, the difference better represents thermal operating conditions.

In [11]:
df["Mechanical Power"] = (
    df["Rotational speed [rpm]"]
    * df["Torque [Nm]"]
)

#Why? Machines under higher mechanical load are generally more susceptible to wear and failure.

In [12]:
df["Wear Rate"] = (
    df["Tool wear [min]"]
    /
    (df["Rotational speed [rpm]"] + 1e-6)
)

#Why? This indicates how much wear accumulates relative to machine speed.

In [13]:
df["Thermal Stress"] = (
    df["Temperature Difference"]
    *
    df["Torque [Nm]"]
)
#Thermal stress combines temperature and mechanical load, both of which significantly influence machine health.

In [14]:
df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,Temperature Difference,Mechanical Power,Wear Rate,Thermal Stress
0,2,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,10.5,66382.8,0.000000,449.40
1,1,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,10.5,65190.4,0.002131,486.15
2,1,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,10.4,74001.2,0.003338,513.76
3,1,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,10.4,56603.5,0.004885,410.80
4,1,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,10.5,56320.0,0.006392,420.00


In [15]:
df.columns

Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
       'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF',
       'Temperature Difference', 'Mechanical Power', 'Wear Rate',
       'Thermal Stress'],
      dtype='str')

In [16]:
df.isnull().sum()

Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
Temperature Difference     0
Mechanical Power           0
Wear Rate                  0
Thermal Stress             0
dtype: int64

In [17]:
df.to_csv(
    "../dataset/ai4i2020_processed.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!


# Notebook Summary

✔ Loaded the AI4I 2020 dataset.

✔ Removed unnecessary identifier columns.

✔ Encoded categorical machine types.

✔ Created four engineered features:

- Temperature Difference
- Mechanical Power
- Wear Rate
- Thermal Stress

✔ Verified the transformed dataset.

✔ Saved the processed dataset for machine learning.